# EvalForge: real GPU training
Enable a CUDA GPU and internet in Kaggle. Add the original Spider tables.json and SQLite databases as an input dataset. No results are pre-filled. Clone the repository below, then set the paths in the preparation cell. Store optional HF_TOKEN / WANDB_API_KEY using Kaggle Secrets and export them without printing their values.


In [ ]:
!git clone https://github.com/Navaneeth-007/EvalForge.git /kaggle/working/EvalForge
%cd /kaggle/working/EvalForge
%pip install -r requirements-ml.txt


In [ ]:
import torch
assert torch.cuda.is_available(), "Enable a CUDA GPU before continuing"
print(torch.cuda.get_device_name(0))


Adjust paths to match the Spider dataset mounted in your notebook. The tables metadata is required; the Hugging Face question rows do not contain schemas.


In [ ]:
from train.dataset_prep import prepare_dataset
prepare_dataset(tables_path="/kaggle/input/spider/tables.json")


In [ ]:
from train.train import train
trainer = train(use_4bit=True)
del trainer
import gc
gc.collect()
torch.cuda.empty_cache()


In [ ]:
import os
os.environ["SPIDER_DB_DIR"] = "/kaggle/input/spider/database"
from eval.run_eval import run_eval
report = run_eval(base_model="microsoft/Phi-3-mini-4k-instruct", finetuned_model=None,
                  adapter_dir="outputs/phi3-sql-qlora/final",
                  eval_samples_path="data/processed/eval_samples.json", n=200,
                  output_path="outputs/eval_results.json", use_4bit=True, skip_bertscore=True)


Download the adapter, training_manifest.json, logs, and eval_results.json as evidence. Review failures and update the model card before publishing. See docs/TRAINING.md for the explicit Hub upload command.
